# 🧠 Knowledge Distillation: Qwen2.5-7B on 2×T4 (fp16 + DDP)

**Teacher:** GPT-4 (via OpenHermes-2.5)  
**Student:** Qwen2.5-7B-Instruct  
**Eval:** ARC-Challenge (compare against baseline)

---

### ⚠️ Before running
1. `Settings → Accelerator` → **GPU T4 x2**
2. `Settings → Internet` → **On**
4. Run the cells in order

**Estimated time:** 3–5 hours 


In [1]:
# ─────────────────────────────────────────────
# CELDA 1 — Verificar entorno y GPUs
# ─────────────────────────────────────────────
import os
import gc
import torch

print(f"🖥️  GPUs disponibles: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"   GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB total | {free/1e9:.1f} GB libres")

assert torch.cuda.device_count() >= 2, "❌ Necesitás 2 GPUs. Activá T4x2 en Settings."
print("\n✅ Entorno listo para DDP")

🖥️  GPUs disponibles: 2
   GPU 0: Tesla T4 — 15.6 GB total | 15.5 GB libres
   GPU 1: Tesla T4 — 15.6 GB total | 15.5 GB libres

✅ Entorno listo para DDP


In [2]:
import sys, os

os.system(f"{sys.executable} -m pip install -q --force-reinstall --no-deps "
          "transformers==4.46.3 "
          "tokenizers==0.20.3 "
          "huggingface-hub==0.26.5 "
          "trl==0.11.4 "
          "accelerate==0.27.2 "   # ← versión sin el upcast forzado
          "peft==0.13.2 ")

os.system(f"{sys.executable} -m pip install -q lm-eval bitsandbytes datasets")

import importlib
for pkg in ["transformers", "tokenizers", "huggingface_hub", "trl", "accelerate", "peft"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"✅ {pkg}=={mod.__version__}")
    except Exception as e:
        print(f"❌ {pkg}: {e}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.8/447.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.

2026-03-07 18:18:06.755005: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772907487.142967      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772907487.264068      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772907488.210387      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772907488.210440      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772907488.210442      55 computation_placer.cc:177] computation placer alr

✅ peft==0.13.2


In [3]:
import os

accelerate_config = """compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
mixed_precision: 'fp16'
num_machines: 1
num_processes: 2
"""

os.makedirs("/root/.cache/huggingface/accelerate", exist_ok=True)
with open("/root/.cache/huggingface/accelerate/default_config.yaml", "w") as f:
    f.write(accelerate_config)

print("✅ Accelerate config: DDP, mixed precision fp16")

✅ Accelerate config: DDP, mixed precision fp16


In [4]:
# ─────────────────────────────────────────────
# CELDA 4 — Cargar y preprocesar OpenHermes-2.5
# ─────────────────────────────────────────────
from datasets import load_dataset

print("📦 Cargando OpenHermes-2.5 (puede tardar ~2 min)...")

dataset = load_dataset(
    "teknium/OpenHermes-2.5",
    split="train",
    streaming=False
)

# ⚡ Cambiá a 5_000 para test rápido (~40 min)
N_SAMPLES = 15_000
dataset = dataset.shuffle(seed=42).select(range(N_SAMPLES))

print(f"✅ Dataset cargado: {len(dataset):,} ejemplos")
print(f"📋 Columnas: {dataset.column_names}")
print(f"\n🔍 Ejemplo:")
print(dataset[0])

📦 Cargando OpenHermes-2.5 (puede tardar ~2 min)...


README.md: 0.00B [00:00, ?B/s]

openhermes2_5.json:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1001551 [00:00<?, ? examples/s]

✅ Dataset cargado: 15,000 ejemplos
📋 Columnas: ['custom_instruction', 'topic', 'model_name', 'model', 'skip_prompt_formatting', 'category', 'conversations', 'views', 'language', 'id', 'title', 'idx', 'hash', 'avatarUrl', 'system_prompt', 'source']

🔍 Ejemplo:
{'custom_instruction': None, 'topic': None, 'model_name': None, 'model': None, 'skip_prompt_formatting': None, 'category': None, 'conversations': [{'from': 'human', 'value': 'A Sierpinski arrowhead curve is created by dividing an equilateral triangle into three smaller equilateral triangles and removing the central one, then doing the same for each remaining smaller triangle. If the length of the original triangle is 6cm, what is the perimeter of the Sierpinski arrowhead curve after the third iteration?', 'weight': None}, {'from': 'gpt', 'value': "After each iteration, the perimeter of the Sierpinski arrowhead curve increases by a factor of 2. Let's calculate the perimeter after the third iteration:\n\n1st iteration:\nPerimeter = 

In [5]:
# ─────────────────────────────────────────────
# CELDA 5 — Formatear en ChatML (formato nativo de Qwen2.5)
# ─────────────────────────────────────────────
def format_openhermes(example):
    """
    OpenHermes tiene conversaciones en formato:
    [{"from": "system", "value": ...},
     {"from": "human", "value": ...},
     {"from": "gpt", "value": ...}]

    Lo convertimos a ChatML (formato nativo de Qwen2.5):
    <|im_start|>system\n...<|im_end|>
    <|im_start|>user\n...<|im_end|>
    <|im_start|>assistant\n...<|im_end|>
    """
    conversations = example.get("conversations", [])
    system_msg = ""
    user_msg = ""
    assistant_msg = ""

    for turn in conversations:
        role  = turn.get("from", "")
        value = turn.get("value", "")
        if role == "system":   system_msg    = value
        elif role == "human":  user_msg      = value
        elif role == "gpt":    assistant_msg = value

    text = ""
    if system_msg:
        text += f"<|im_start|>system\n{system_msg}<|im_end|>\n"
    text += f"<|im_start|>user\n{user_msg}<|im_end|>\n"
    text += f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
    return {"text": text}


dataset = dataset.map(
    format_openhermes,
    remove_columns=dataset.column_names,
    num_proc=4
)
dataset = dataset.filter(lambda x: len(x["text"]) > 50)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

print(f"✅ Dataset formateado")
print(f"📊 Train: {len(train_dataset):,} | Val: {len(eval_dataset):,}")
print(f"\n🔍 Ejemplo formateado:")
print(train_dataset[0]["text"][:500])
print("...")

Map (num_proc=4):   0%|          | 0/15000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15000 [00:00<?, ? examples/s]

✅ Dataset formateado
📊 Train: 14,250 | Val: 750

🔍 Ejemplo formateado:
<|im_start|>system
You are an AI assistant. You should describe the task and explain your answer. While answering a multiple choice question, first output the correct answer(s). Then explain why other answers are wrong. You might need to use additional knowledge to answer the question.<|im_end|>
<|im_start|>user
Are these paraphrases?
An article by Richard Prum in the `` Houston Chronicle '' published 18 December 2005 presented Dina Cappiello 's position as follows :
In an article by Richard Pru
...


In [6]:
import torch
from transformers import AutoTokenizer

MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "/kaggle/working/qwen25-7b-distilled"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True, padding_side="right"
)

# ⚠️ FIX: eos_token en Qwen2.5 = <|im_end|>, que aparece en cada turno ChatML.
# Usarlo como pad_token hace que el collator enmascare todos los labels → loss=0.0
tokenizer.pad_token = "<|endoftext|>"

print(f"✅ Tokenizer listo")
print(f"   pad_token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"   eos_token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer listo
   pad_token: <|endoftext|> (id=151643)
   eos_token: <|im_end|> (id=151645)


In [7]:
# ─────────────────────────────────────────────
# CELDA 7 — Guardar dataset en disco para DDP
# ─────────────────────────────────────────────
# Los procesos DDP corren como subprocesos separados,
# necesitan leer el dataset desde disco

DATA_PATH = "/kaggle/working/openhermes_formatted"
dataset.save_to_disk(DATA_PATH)
print(f"✅ Dataset guardado en {DATA_PATH}")

Saving the dataset (0/1 shards):   0%|          | 0/14250 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/750 [00:00<?, ? examples/s]

✅ Dataset guardado en /kaggle/working/openhermes_formatted


In [8]:
# ─────────────────────────────────────────────
# CELDA 7b — Pre-descargar modelo (evita race condition DDP)
# ─────────────────────────────────────────────
from huggingface_hub import snapshot_download
import torch, gc

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print("📥 Descargando archivos del modelo a caché local...")
print("   (sin cargar en memoria ni tocar VRAM)")

snapshot_download(
    repo_id=MODEL_ID,
    repo_type="model",
    ignore_patterns=["*.msgpack", "*.h5", "flax_*"],
)

print("✅ Modelo en caché — los workers DDP leerán desde disco")
print(f"   VRAM libre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")

📥 Descargando archivos del modelo a caché local...
   (sin cargar en memoria ni tocar VRAM)


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

✅ Modelo en caché — los workers DDP leerán desde disco
   VRAM libre: 15.5 GB / 15.6 GB


In [9]:
train_script = '''
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_from_disk

MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH  = "/kaggle/working/openhermes_formatted"
OUTPUT_DIR = "/kaggle/working/qwen25-7b-distilled"

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

def main():
    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID, trust_remote_code=True, padding_side="right"
    )
    tokenizer.pad_token = "<|endoftext|>"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        trust_remote_code=True,
        device_map={"": local_rank},
        attn_implementation="sdpa",
    )

    model = prepare_model_for_kbit_training(model)
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.config.use_cache = False

    if local_rank == 0:
        model.print_trainable_parameters()

    dataset    = load_from_disk(DATA_PATH)
    train_data = dataset["train"]
    eval_data  = dataset["test"]

    sft_config = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=2,     # ✅ 4→2 — menos logits en memoria
        per_device_eval_batch_size=1,      # ✅ 2→1 — idem para eval
        gradient_accumulation_steps=8,     # ✅ 4→8 — batch efectivo = 2*8*2 = 32
        learning_rate=2e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=True,
        bf16=False,
        max_grad_norm=0.3,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        logging_steps=2,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_seq_length=512,               
        packing=True,
        dataset_text_field="text",
        report_to="none",
        dataloader_num_workers=4,
        ddp_find_unused_parameters=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_data,
        eval_dataset=eval_data,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    if local_rank == 0:
        sample_batch = next(iter(trainer.get_train_dataloader()))
        labels = sample_batch["labels"]
        valid  = (labels != -100).sum().item()
        total  = labels.numel()
        print(f"\\n🔍 Sanity check — labels válidos: {valid}/{total} ({valid/total*100:.1f}%)")
        if valid == 0:
            raise ValueError("❌ TODOS los labels son -100!")

        batch_efectivo = sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps * 2
        steps_totales  = len(train_data) // batch_efectivo
        print(f"\\n🚀 Iniciando fine-tuning QLoRA en 2xT4...")
        print(f"   Batch efectivo: {batch_efectivo} | Steps: {steps_totales}")
        print(f"   ⚡ Packing=True — steps reales serán menos\\n")

    trainer.train()

    if local_rank == 0:
        print("\\n💾 Guardando adapters LoRA...")
        trainer.model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"\\n✅ Adapters guardados en {OUTPUT_DIR}")

if __name__ == "__main__":
    main()
'''

with open("/kaggle/working/train_ddp.py", "w") as f:
    f.write(train_script)

print("✅ train_ddp.py guardado")

✅ train_ddp.py guardado


In [ ]:
import os, shutil, gc, torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ✅ Limpiar TODA la VRAM antes de lanzar DDP
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"   GPU {i}: {free/1e9:.1f} GB libres / {total/1e9:.1f} GB totales")

ckpt_dir = "/kaggle/working/qwen25-7b-distilled"
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print(f"🧹 Output dir limpiado: {ckpt_dir}")

print("🚀 Lanzando entrenamiento con accelerate + DDP...\n")
exit_code = os.system("accelerate launch /kaggle/working/train_ddp.py")

if exit_code == 0:
    print("\n✅ Entrenamiento completado exitosamente")
else:
    print(f"\n❌ Error (exit code: {exit_code}) — revisá los logs arriba")

In [11]:
# ─────────────────────────────────────────────
# CELDA 9b — Merge LoRA al modelo base (celda separada para evitar OOM)
# ─────────────────────────────────────────────
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID    = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_DIR = "/kaggle/working/qwen25-7b-distilled"
OUTPUT_DIR  = "/kaggle/working/qwen25-7b-destilled-gpt4"

# Liberar memoria de procesos DDP anteriores
gc.collect()
torch.cuda.empty_cache()
print(f"🧹 VRAM libre antes del merge: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

print("📥 Cargando modelo base en fp16...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("🔀 Cargando y mergeando adapters LoRA...")
model_merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model_merged = model_merged.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)

print(f"💾 Guardando modelo mergeado en {OUTPUT_DIR}...")
model_merged.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Merge completado → {OUTPUT_DIR}")

🧹 VRAM libre antes del merge: 15.4 GB
📥 Cargando modelo base en fp16...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔀 Cargando y mergeando adapters LoRA...
💾 Guardando modelo mergeado en /kaggle/working/qwen25-7b-finetunning...
✅ Merge completado → /kaggle/working/qwen25-7b-finetunning


In [18]:
# ─────────────────────────────────────────────
# CELDA 10 — Guardar modelo en Hugging Face Hub
# ─────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO_ID    = "Cukinator/qwen25-7b-destilled-gpt4"
OUTPUT_DIR = "/kaggle/working/qwen25-7b-destilled-gpt4"

api = HfApi()

# Crear el repo si no existe
api.create_repo(repo_id=REPO_ID, private=True, exist_ok=True)

print(f"📤 Subiendo archivos desde {OUTPUT_DIR}...")
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"✅ Guardado en https://huggingface.co/{REPO_ID}")

📤 Subiendo archivos desde /kaggle/working/qwen25-7b-finetunning...


model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Upload 4 LFS files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

✅ Guardado en https://huggingface.co/Cukinator/qwen25-7b-finetunning


In [12]:
# ─────────────────────────────────────────────
# CELDA 11 — Verificar archivos guardados
# ─────────────────────────────────────────────
import os

OUTPUT_DIR = "/kaggle/working/qwen25-7b-destilled-gpt4"
files = os.listdir(OUTPUT_DIR) if os.path.exists(OUTPUT_DIR) else []

print(f"📁 Archivos en {OUTPUT_DIR}:")
total_gb = 0
for f in sorted(files):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    total_gb += size_mb / 1000
    print(f"   {f:45s} {size_mb:>8.1f} MB")
print(f"\n   Total: {total_gb:.2f} GB")

📁 Archivos en /kaggle/working/qwen25-7b-finetunning:
   added_tokens.json                                  0.0 MB
   config.json                                        0.0 MB
   generation_config.json                             0.0 MB
   merges.txt                                         1.7 MB
   model-00001-of-00004.safetensors                4877.7 MB
   model-00002-of-00004.safetensors                4932.8 MB
   model-00003-of-00004.safetensors                4330.9 MB
   model-00004-of-00004.safetensors                1090.0 MB
   model.safetensors.index.json                       0.0 MB
   special_tokens_map.json                            0.0 MB
   tokenizer.json                                    11.4 MB
   tokenizer_config.json                              0.0 MB
   vocab.json                                         2.8 MB

   Total: 15.25 GB


# Se recomienda reiniciar aca

In [4]:
# ─────────────────────────────────────────────
# CELDA 12 — Cargar modelo desde Hugging Face Hub
# ─────────────────────────────────────────────
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO_ID = "Cukinator/qwen25-7b-destilled-gpt4"

print(f"📥 Cargando modelo desde {REPO_ID}...")
tokenizer_eval = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
model_eval = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model_eval.eval()
print("✅ Modelo cargado correctamente")
print(f"   VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB")

📥 Cargando modelo desde Cukinator/qwen25-7b-destilled-gpt4...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Modelo cargado correctamente
   VRAM usada: 6.7 GB


In [6]:
# ─────────────────────────────────────────────
# CELDA 13 — Evaluar con ARC-Challenge
# ─────────────────────────────────────────────
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import lm_eval
from lm_eval.models.huggingface import HFLM

print("\n🧠 Evaluando ARC-Challenge (100 ejemplos, zero-shot)...")
hflm = HFLM(
    pretrained=model_eval,
    tokenizer=tokenizer_eval,
    batch_size=4,
    device="cuda:0",  # ← forzar GPU
)
results = lm_eval.simple_evaluate(
    model=hflm,
    tasks=["arc_challenge"],
    num_fewshot=0,
    limit=100
)
acc      = results["results"]["arc_challenge"]["acc,none"]
acc_norm = results["results"]["arc_challenge"]["acc_norm,none"]
BASELINE = 0.46
print("\n" + "="*50)
print("📊 RESULTADOS FINALES")
print("="*50)
print(f"  Baseline (antes de destilar):  {BASELINE*100:.1f}%")
print(f"  Destilado — acc:               {acc*100:.1f}%")
print(f"  Destilado — acc_norm:          {acc_norm*100:.1f}%")
print(f"  Mejora acc_norm:               {(acc_norm - BASELINE)*100:+.1f}%")
print("="*50)


🧠 Evaluando ARC-Challenge (100 ejemplos, zero-shot)...


Running loglikelihood requests: 100%|██████████| 400/400 [00:13<00:00, 30.37it/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).



📊 RESULTADOS FINALES
  Baseline (antes de destilar):  46.0%
  Destilado — acc:               43.0%
  Destilado — acc_norm:          40.0%
  Mejora acc_norm:               -6.0%


In [7]:
# ─────────────────────────────────────────────
# CELDA 14 — (Bonus) Probar el modelo en vivo
# ─────────────────────────────────────────────
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model=model_eval,
    tokenizer=tokenizer_eval,
    torch_dtype=torch.float16,
    device_map="auto"
)

prompts = [
    "¿Cuál es la diferencia entre fusión nuclear y fisión nuclear?",
    "Explain gradient descent in simple terms.",
    "What causes the seasons on Earth?"
]

print("🗣️ Respuestas del modelo destilado:\n")
for prompt in prompts:
    messages = [{"role": "user", "content": prompt}]
    output   = pipe(
        messages,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer_eval.eos_token_id
    )
    response = output[0]["generated_text"][-1]["content"]
    print(f"❓ {prompt}")
    print(f"💬 {response}")
    print("-" * 60)

🗣️ Respuestas del modelo destilado:

❓ ¿Cuál es la diferencia entre fusión nuclear y fisión nuclear?
💬 La fusión nuclear y la fisión nuclear son dos procesos de producción de energía que se basan en la transformación de materia. Sin embargo, estos procesos son muy diferentes en términos de cómo ocurren y cuáles son sus aplicaciones.

1. Fisión Nuclear: Este proceso se refiere a la descomposición de un núcleo atómico pesado (como el uranio o el plutonio) en dos núcleos más pequeños, liberando energía en el proceso. Este tipo de reacción es la base para los reactores nucleares convencionales y las armas nucleares.

2. Fusión Nuclear: En este caso, dos núcleos atómicos ligeros (generalmente hidrógeno o su isótopo deutérico) se combinan para formar un núcleo más pesado, también liberando energía en el proceso. Este tipo de reacción es la fuente de
------------------------------------------------------------
❓ Explain gradient descent in simple terms.
💬 Gradient descent is an optimization a